In [1]:
!nvidia-smi

Mon Apr 14 07:06:10 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install --upgrade pip
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

In [4]:
import zipfile

with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

In [5]:
data = pd.read_csv("Travel.csv")

In [6]:
data.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [7]:
data['TypeofContact'].value_counts()

,count
TypeofContact,
Self Enquiry,3444
Company Invited,1419


In [8]:
data['Gender'].value_counts()

,count
Gender,
Male,2916
Female,1817
Fe Male,155


In [9]:
data['Gender'] = data["Gender"].replace('Fe Male', "Female")

In [10]:
data['Gender'].value_counts()

,count
Gender,
Male,2916
Female,1972


In [11]:
data['ProductPitched'].value_counts()

,count
ProductPitched,
Basic,1842
Deluxe,1732
Standard,742
Super Deluxe,342
King,230


In [12]:
data['MaritalStatus'].value_counts()

,count
MaritalStatus,
Married,2340
Divorced,950
Single,916
Unmarried,682


In [13]:
data['MaritalStatus'] = data["MaritalStatus"].replace('Single', "Unmarried")

In [14]:
data['MaritalStatus'].value_counts()

,count
MaritalStatus,
Married,2340
Unmarried,1598
Divorced,950


In [15]:
data['Designation'].value_counts()

,count
Designation,
Executive,1842
Manager,1732
Senior Manager,742
AVP,342
VP,230


In [16]:
data.isnull().sum()

,0
CustomerID,0
ProdTaken,0
Age,226
TypeofContact,25
CityTier,0
DurationOfPitch,251
Occupation,0
Gender,0
NumberOfPersonVisiting,0
NumberOfFollowups,45


In [17]:
null_features = [features for features in data.columns if data[features].isnull().sum()>=1]
for feature in null_features:
  print(feature, np.round(data[feature].isnull().mean()*100,5), '% missing values')

Age 4.62357 % missing values
TypeofContact 0.51146 % missing values
DurationOfPitch 5.13502 % missing values
NumberOfFollowups 0.92062 % missing values
PreferredPropertyStar 0.53191 % missing values
NumberOfTrips 2.86416 % missing values
NumberOfChildrenVisiting 1.35025 % missing values
MonthlyIncome 4.76678 % missing values


In [18]:
data[null_features].select_dtypes(exclude='object').describe()

,Age,DurationOfPitch,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,NumberOfChildrenVisiting,MonthlyIncome
count,4662.000000,4637.000000,4843.000000,4862.000000,4748.000000,4822.000000,4655.000000
mean,37.622265,15.490835,3.708445,3.581037,3.236521,1.187267,23619.853491
std,9.316387,8.519643,1.002509,0.798009,1.849019,0.857861,5380.698361
min,18.000000,5.000000,1.000000,3.000000,1.000000,0.000000,1000.000000
25%,31.000000,9.000000,3.000000,3.000000,2.000000,1.000000,20346.000000
50%,36.000000,13.000000,4.000000,3.000000,3.000000,1.000000,22347.000000
75%,44.000000,20.000000,4.000000,4.000000,4.000000,2.000000,25571.000000
max,61.000000,127.000000,6.000000,5.000000,22.000000,3.000000,98678.000000


In [19]:
data['Age'].fillna(data['Age'].median(),inplace = True)

In [20]:
data['TypeofContact'].fillna(data['TypeofContact'].mode()[0],inplace = True)

In [21]:
data['DurationOfPitch'].fillna(data['DurationOfPitch'].median(),inplace = True)

In [22]:
data['NumberOfFollowups'].fillna(data['NumberOfFollowups'].mode()[0],inplace = True)

In [23]:
data['PreferredPropertyStar'].fillna(data['PreferredPropertyStar'].mode()[0],inplace = True)

In [24]:
data['NumberOfTrips'].fillna(data['NumberOfTrips'].median(),inplace = True)

In [25]:
data['NumberOfChildrenVisiting'].fillna(data['NumberOfChildrenVisiting'].mode()[0],inplace = True)

In [26]:
data['MonthlyIncome'].fillna(data['MonthlyIncome'].median(),inplace = True)

In [27]:
data.isnull().sum()

,0
CustomerID,0
ProdTaken,0
Age,0
TypeofContact,0
CityTier,0
DurationOfPitch,0
Occupation,0
Gender,0
NumberOfPersonVisiting,0
NumberOfFollowups,0


In [28]:
data.drop('CustomerID', inplace = True , axis=1)

In [29]:
data['TotalVisiting'] = data['NumberOfChildrenVisiting'] + data['NumberOfPersonVisiting']
data.drop(columns=['NumberOfChildrenVisiting','NumberOfPersonVisiting'],inplace=True, axis =1)

In [30]:
data.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisiting
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,0,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0


In [31]:
from sklearn.model_selection import train_test_split

X = data.drop('ProdTaken', axis=1)
y = data['ProdTaken']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
X_train.shape, X_test.shape

((3910, 17), (978, 17))

In [33]:
categorical_features = X.select_dtypes(include='object').columns
numerical_features = X.select_dtypes(exclude='object').columns

In [34]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer ##--> for combining multiple transformation technique(i always forgot this...)

numeric_transformer = StandardScaler()
onehot_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", onehot_transformer, categorical_features),
         ("StandardScaler", numeric_transformer, numerical_features),
    ]
)

In [35]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [36]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

In [37]:
%%time

model = RandomForestClassifier(n_estimators=100,min_samples_split=2,max_features= 8, max_depth = 15, n_jobs = -1)
model.fit(X_train, y_train)

# y_train_pred = model.predict(X_train)
# y_test_pred = model.predict(X_test)

# model_train_accuracy = accuracy_score(y_train, y_train_pred)
# model_train_precision = precision_score(y_train, y_train_pred)
# model_train_recall = recall_score(y_train, y_train_pred)
# model_train_f1 = f1_score(y_train, y_train_pred, average='weighted')
# model_train_roc_auc = roc_auc_score(y_train, y_train_pred)

# model_test_accuracy = accuracy_score(y_test, y_test_pred)
# model_test_precision = precision_score(y_test, y_test_pred)
# model_test_recall = recall_score(y_test, y_test_pred)
# model_test_f1 = f1_score(y_test, y_test_pred, average='weighted')
# model_test_roc_auc = roc_auc_score(y_test, y_test_pred)

CPU times: user 762 ms, sys: 40.2 ms, total: 802 ms
Wall time: 503 ms


RandomForestClassifier(max_depth=15, max_features=8, n_jobs=-1)

In [38]:
# print('Model performance Train set')
# print("- Accuracy: {:.4f}".format(model_train_accuracy))
# print('- Precision: {:.4f}'.format(model_train_precision))
# print('- Recall: {:.4f}'.format(model_train_recall))
# print('- F1 Score: {:.4f}'.format(model_train_f1))
# print('- ROC AUC Score: {:.4f}'.format(model_train_roc_auc))

# print('----------------------------------')

# print('Model performance Test set')
# print("- Accuracy: {:.4f}".format(model_test_accuracy))
# print('- Precision: {:.4f}'.format(model_test_precision))
# print('- Recall: {:.4f}'.format(model_test_recall))
# print('- F1 Score: {:.4f}'.format(model_test_f1))
# print('- ROC AUC Score: {:.4f}'.format(model_test_roc_auc))

In [39]:
%%time

model = RandomForestClassifier(n_estimators=500,min_samples_split=2,max_features= 8, max_depth = 15, n_jobs = -1)
model.fit(X_train, y_train)

# y_train_pred = model.predict(X_train)
# y_test_pred = model.predict(X_test)

CPU times: user 4 s, sys: 63.4 ms, total: 4.06 s
Wall time: 4.45 s


RandomForestClassifier(max_depth=15, max_features=8, n_estimators=500,
                       n_jobs=-1)

In [40]:
%%time

model = RandomForestClassifier(n_estimators=1000,min_samples_split=2,max_features= 8, max_depth = 30, n_jobs = -1)
model.fit(X_train, y_train)

# y_train_pred = model.predict(X_train)
# y_test_pred = model.predict(X_test)

# model_train_accuracy = accuracy_score(y_train, y_train_pred)
# model_train_precision = precision_score(y_train, y_train_pred)
# model_train_recall = recall_score(y_train, y_train_pred)
# model_train_f1 = f1_score(y_train, y_train_pred, average='weighted')
# model_train_roc_auc = roc_auc_score(y_train, y_train_pred)

# model_test_accuracy = accuracy_score(y_test, y_test_pred)
# model_test_precision = precision_score(y_test, y_test_pred)
# model_test_recall = recall_score(y_test, y_test_pred)
# model_test_f1 = f1_score(y_test, y_test_pred, average='weighted')
# model_test_roc_auc = roc_auc_score(y_test, y_test_pred)

CPU times: user 8.03 s, sys: 120 ms, total: 8.15 s
Wall time: 11.1 s


RandomForestClassifier(max_depth=30, max_features=8, n_estimators=1000,
                       n_jobs=-1)

In [41]:
# print('Model performance Train set')
# print("- Accuracy: {:.4f}".format(model_train_accuracy))
# print('- Precision: {:.4f}'.format(model_train_precision))
# print('- Recall: {:.4f}'.format(model_train_recall))
# print('- F1 Score: {:.4f}'.format(model_train_f1))
# print('- ROC AUC Score: {:.4f}'.format(model_train_roc_auc))

# print('----------------------------------')

# print('Model performance Test set')
# print("- Accuracy: {:.4f}".format(model_test_accuracy))
# print('- Precision: {:.4f}'.format(model_test_precision))
# print('- Recall: {:.4f}'.format(model_test_recall))
# print('- F1 Score: {:.4f}'.format(model_test_f1))
# print('- ROC AUC Score: {:.4f}'.format(model_test_roc_auc))

In [42]:
## cuML's accelerator mode

In [43]:
%load_ext cuml.accel

[2025-04-14 07:06:37.617] [CUML] [info] cuML: Installed accelerator for sklearn.
[2025-04-14 07:06:48.957] [CUML] [info] cuML: Installed accelerator for umap.
[2025-04-14 07:06:48.967] [CUML] [info] cuML: Installed accelerator for hdbscan.
[2025-04-14 07:06:48.967] [CUML] [info] cuML: Successfully initialized accelerator.


In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

In [45]:
%%time

model = RandomForestClassifier(n_estimators=100,min_samples_split=2,max_features= 8, max_depth = 15, n_jobs = -1)
model.fit(X_train, y_train)

CPU times: user 301 ms, sys: 167 ms, total: 467 ms
Wall time: 379 ms


RandomForestClassifier(max_depth=15, max_features=8, n_jobs=-1)

In [46]:
%%time

model = RandomForestClassifier(n_estimators=500,min_samples_split=2,max_features= 8, max_depth = 15, n_jobs = -1)
model.fit(X_train, y_train)

# y_train_pred = model.predict(X_train)
# y_test_pred = model.predict(X_test)

CPU times: user 757 ms, sys: 349 ms, total: 1.11 s
Wall time: 733 ms


RandomForestClassifier(max_depth=15, max_features=8, n_estimators=500,
                       n_jobs=-1)

In [47]:
# %%time

# clf = RandomForestClassifier(n_estimators=500,min_samples_split=2,max_features= 8, max_depth = 15, n_jobs=-1)
# clf.fit(X_train, y_train)

In [48]:
y_pred = model.predict(X_test)
cr = classification_report(y_test, y_pred)
print(cr)

              precision    recall  f1-score   support

           0       0.93      0.99      0.96       787
           1       0.96      0.70      0.81       191

    accuracy                           0.94       978
   macro avg       0.95      0.84      0.88       978
weighted avg       0.94      0.94      0.93       978



In [49]:
%%time

model = RandomForestClassifier(n_estimators=1000, max_depth=30, max_features= 8,min_samples_split=2, n_jobs=-1)
model.fit(X_train, y_train)

CPU times: user 1.89 s, sys: 759 ms, total: 2.65 s
Wall time: 1.73 s


RandomForestClassifier(max_depth=30, max_features=8, n_estimators=1000,
                       n_jobs=-1)

In [50]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.99      0.96       787
           1       0.96      0.71      0.81       191

    accuracy                           0.94       978
   macro avg       0.95      0.85      0.89       978
weighted avg       0.94      0.94      0.93       978

